In [4]:
import zipfile
import os

# Unzip the audio dataset
with zipfile.ZipFile('Main_set.zip', 'r') as zip_ref:
    zip_ref.extractall('Main_set')

print("Extraction complete! Extracted files:", len(os.listdir('Main_set')))

Extraction complete! Extracted files: 1


In [5]:
import os
import re
import glob
import pandas as pd
import librosa

# Parse filenames helper
def parse_filename(filename):
    base_name = os.path.basename(filename)
    match = re.match(r'M(\d+)_(\d+)\.wav', base_name, re.IGNORECASE)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None

# Load experiments table
df_exp = pd.read_excel('experiments.xlsx')
print("Experiments table loaded:", df_exp.shape)

# Scan audio files
audio_files = glob.glob('Main_set/**/*.wav', recursive=True) + glob.glob('Main_set/*.wav', recursive=True)
print(f"Total audio files found: {len(audio_files)}")

file_records = []
for fpath in audio_files:
    ch, exp_num = parse_filename(fpath)
    if ch is not None:
        y, sr = librosa.load(fpath, sr=None)
        duration = librosa.get_duration(y=y, sr=sr)
        file_records.append({
            'file': os.path.basename(fpath),
            'file_path': fpath,
            'channel': ch,
            'exp': exp_num,
            'duration_s': duration,
            'sr': sr
        })

df_files = pd.DataFrame(file_records)

# Merge with experiments metadata
Records = pd.merge(df_files, df_exp, on='exp', how='left')
print("Records register created successfully!")
display(Records.head())

Experiments table loaded: (31, 8)
Total audio files found: 107
Records register created successfully!


,file,file_path,channel,exp,duration_s,sr,instrument,place,tuning,peg_angle_deg,operator,room,gain_db
0,M3_11.wav,Main_set/Main_set/M3_11.wav,3,11,84.017687,22050,акустическая (нейлон),1,detuned,180°,оператор_1,студия_1,0
1,M1_14.wav,Main_set/Main_set/M1_14.wav,1,14,94.439592,22050,укулеле,место 1,Расстроен,-180°,оператор_1,студия_1,0
2,M2_05.wav,Main_set/Main_set/M2_05.wav,2,5,101.928027,22050,акуст. нейлон,место 1,настроен,NaN,оператор_1,студия_1,0
3,M2_21.wav,Main_set/Main_set/M2_21.wav,2,21,87.639637,22050,укулеле,место 3,Расстроен,180 град.,оператор_1,студия_1,0
4,M1_07.wav,Main_set/Main_set/M1_07.wav,1,7,184.380454,22050,полуакустическая гитара,Место 3,tuned,NaN,оператор_1,студия_1,0


In [7]:
import numpy as np
import librosa

windows_data = []

print("Starting optimized feature extraction...")
for idx, row in Records.iterrows():
    y, sr = librosa.load(row['file_path'], sr=22050)
    window_size = sr  # 1 second

    for start_sample in range(0, len(y), window_size):
        end_sample = start_sample + window_size
        window_audio = y[start_sample:end_sample]

        if len(window_audio) < sr * 0.5:
            continue

        start_s = start_sample / sr
        rms = float(np.sqrt(np.mean(window_audio**2)))
        mfccs = librosa.feature.mfcc(y=window_audio, sr=sr, n_mfcc=13).mean(axis=1).tolist()

        # Fast YIN pitch estimation instead of slow PYIN
        f0 = librosa.yin(window_audio, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
        f0_mean = float(np.nanmean(f0)) if not np.isnan(f0).all() else 0.0

        windows_data.append({
            'file': row['file'],
            'channel': row['channel'],
            'exp': row['exp'],
            'start_s': start_s,
            'rms': rms,
            'f0_mean': f0_mean,
            'mfcc': mfccs,
            'instrument': row.get('instrument'),
            'tuning': row.get('tuning'),
            'place': row.get('place')
        })

df_windows = pd.DataFrame(windows_data)
df_windows.to_csv('Windows_dataset.csv', index=False)
print("Windows_dataset.csv created and saved successfully!")
display(df_windows.head())

Starting optimized feature extraction...
Windows_dataset.csv created and saved successfully!


,file,channel,exp,start_s,rms,f0_mean,mfcc,instrument,tuning,place
0,M3_11.wav,3,11,0.0,0.000092,95.736101,"[-768.9692993164062, 48.88199234008789, 33.523...",акустическая (нейлон),detuned,1
1,M3_11.wav,3,11,1.0,0.003654,195.736709,"[-667.0174560546875, 60.21060562133789, 11.678...",акустическая (нейлон),detuned,1
2,M3_11.wav,3,11,2.0,0.004422,367.475584,"[-553.2125854492188, 84.04924011230469, -14.37...",акустическая (нейлон),detuned,1
3,M3_11.wav,3,11,3.0,0.005013,409.328890,"[-555.5443725585938, 86.37500762939453, -4.001...",акустическая (нейлон),detuned,1
4,M3_11.wav,3,11,4.0,0.005235,444.749263,"[-564.5115966796875, 87.61638641357422, -0.701...",акустическая (нейлон),detuned,1
